# 🏭 Industrial Cortex - Feature Engineering & Unified Telemetry Analysis

This notebook runs the formal feature engineering and column standardization pipeline across our four core datasets:
1. **Large Industrial Pump Maintenance Dataset** (`Large_Industrial_Pump_Maintenance_Dataset.csv`)
2. **Equipment Anomaly Data** (`equipment_anomaly_data.csv`)
3. **AI4I 2020 Predictive Maintenance Dataset** (`ai4i2020.csv`)
4. **IoT Equipment Monitoring Dataset** (`iot_equipment_monitoring_dataset.csv`)

It validates that all datasets have identical columns (raw metrics, normalizations, FFT derivations, and Auto-ML anomaly scores) and verifies the physics-based simulator integration.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Ensure backend is in python path
sys.path.append(os.path.join(os.getcwd(), 'backend'))
sys.path.append(os.getcwd())

print("Libraries and display utility imported successfully!")

## 🧹 1. Execute Feature Engineering & Schema Standardization

In [ ]:
from backend.utils.feature_engineer import standardize_and_engineer

# Run the feature engineering pipeline
standardize_and_engineer()
print("[SUCCESS] Feature engineering and alignment complete!")

## 📂 2. Load Standardized Datasets & Verify Schemas

In [ ]:
df_pump = pd.read_csv("backend/data/vibration/Large_Industrial_Pump_Maintenance_Dataset.csv")
df_anomaly = pd.read_csv("backend/data/vibration/equipment_anomaly_data.csv")
df_ai4i = pd.read_csv("backend/data/vibration/ai4i2020.csv")
df_iot = pd.read_csv("backend/data/vibration/iot_equipment_monitoring_dataset.csv")

print(f"Pump Dataset Shape: {df_pump.shape}")
print(f"Anomaly Dataset Shape: {df_anomaly.shape}")
print(f"AI4I 2020 Dataset Shape: {df_ai4i.shape}")
print(f"IoT Sensor Dataset Shape: {df_iot.shape}")

# Check schema columns alignment
assert list(df_pump.columns) == list(df_anomaly.columns) == list(df_ai4i.columns) == list(df_iot.columns), "ERROR: Column schemas do not match!"
print("\n[SUCCESS] Schema columns are 100% identical across all datasets:")
print(list(df_pump.columns))

## 📊 3. Metric Distribution Across Asset Types
We can now query and inspect features across all asset classes using the exact same code!

In [ ]:
from IPython.display import display
# Concatenate all datasets for uniform analysis
df_all = pd.concat([df_pump, df_anomaly, df_ai4i, df_iot], ignore_index=True)
print(f"Total unified rows: {df_all.shape[0]}")

print("\n--- Mean Metrics by Asset Type ---")
display(df_all.groupby('asset_type')[['temperature', 'vibration', 'pressure', 'voltage', 'current', 'rpm', 'torque', 'Anomaly_Score']].mean())

print("\n--- Anomaly Class Distribution by Dataset ---")
display(df_all.groupby(['source_dataset', 'fault_status']).size().unstack(fill_value=0))

## 🔬 4. Physics-based Vibration Simulator Verification

In [ ]:
from backend.luigi_ears.vibration_tools import VibrationAnalyzer

analyzer = VibrationAnalyzer()

# Test 1: Normal Pump Query
print("\n--- Normal Pump Simulation P-1 ---")
normal_result = analyzer.analyze("P-1", query="How is pump P-1 running?")
print(f"Source Dataset: {normal_result.get('source_dataset')}")
print(f"Vibration RMS: {normal_result.get('vibration_rms'):.2f} mm/s")
print(f"Temperature: {normal_result.get('temperature'):.1f} C")
print(f"Anomaly Score: {normal_result.get('anomaly_score'):.3f}")
print(f"Diagnosis: {normal_result.get('fault_type')}")

# Test 2: Faulty CNC Spindle
print("\n--- Faulty CNC Simulation M14860 ---")
cnc_result = analyzer.analyze("M14860", query="M14860 has a tool wear issue!")
print(f"Source Dataset: {cnc_result.get('source_dataset')}")
print(f"RPM: {cnc_result.get('rpm'):.1f}")
print(f"Torque: {cnc_result.get('torque'):.1f} Nm")
print(f"Tool Wear: {cnc_result.get('tool_wear'):.1f} min")
print(f"Anomaly Score: {cnc_result.get('anomaly_score'):.3f}")
print(f"Diagnosis: {cnc_result.get('fault_type')}")
print(f"Recommendation: {cnc_result.get('recommendation')}")

## 📈 5. Plotting Engineered Features (FFT Peaks)

In [ ]:
# Plot FFT Peaks for Normal vs Anomalous Pump
pump_norm = df_pump[df_pump['fault_status'] == 0].iloc[0]
pump_fault = df_pump[df_pump['fault_status'] == 1].iloc[0]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.array([1, 2])

ax.bar(x - 0.2, [pump_norm['FFT_Feature1'], pump_norm['FFT_Feature2']], width=0.4, color='green', label='Normal Pump')
ax.bar(x + 0.2, [pump_fault['FFT_Feature1'], pump_fault['FFT_Feature2']], width=0.4, color='red', label='Anomalous Pump')

ax.set_xticks(x)
ax.set_xticklabels(['FFT Peak 1 (RPM)', 'FFT Peak 2 (Bearing Fault)'])
ax.set_ylabel('Amplitude')
ax.set_title('Engineered FFT Frequency Peak comparison')
ax.legend()
ax.grid(True)

plt.show()